# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 3: Decoder-only Transformers</font>

# <font color="#003660">Notebook 3: Language Models as Scoring Machines</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... understand that a language model assigns a probability to every text, <br>
        ... can use these probabilities to compare sentences, classify texts without training (zero-shot), and measure perplexity, and <br>
        ... can visualize the attention weights of a decoder model.
    </font>
</div>
</center>
</p>

The following content is inspired by the following sources:

*   Tunstall et al. (2022): Natural Language Processing with Transformers. O'Reilly. https://www.oreilly.com/library/view/natural-language-processing/9781098103231/
*   Hugging Face: LLM Course. https://huggingface.co/learn/llm-course
*   Radford et al. (2019): Language Models are Unsupervised Multitask Learners (the GPT-2 paper). https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

# Language Models Score Texts

In the first two notebooks, we used decoder models to **generate** text: predict a probability distribution over the next token, pick a token, repeat.

But the same model can also be used the other way around. Instead of asking *"what comes next?"*, we can show the model a complete text and ask *"how likely is this text?"*. By the chain rule of probability, the probability of a text is the product of the probabilities of its tokens, each given all previous tokens:

$$P(w_1, w_2, \dots, w_n) = P(w_1) \cdot P(w_2 \mid w_1) \cdot \ldots \cdot P(w_n \mid w_1, \dots, w_{n-1})$$

As in notebook 1, we work with log-probabilities and sum them up instead of multiplying probabilities. The result is a **score** for every text: the higher (i.e., the closer to 0), the more plausible the text is according to the model.

In this notebook, we use this score in three ways:
1. **Comparing sentences:** which of two sentences does the model find more plausible?
2. **Zero-shot classification:** classify texts *without any training*, by scoring each possible label.
3. **Perplexity:** how *surprised* is the model by a text?

Finally, we look inside the model and **visualize the attention weights**.

# Import Packages

As always, we first need to load a number of required Python packages:
- `numpy` and `pandas` for working with arrays and data frames.
- `torch` (PyTorch), the deep learning library on which `transformers` is built.
- `transformers` for loading pre-trained models from the Hugging Face model hub.
- `datasets` for loading datasets from the Hugging Face hub.
- `scikit-learn` for evaluation metrics.
- `matplotlib` for plotting.

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

# Load Pre-trained Model

We use **Qwen2.5-0.5B**, a small (0.5 billion parameters) but surprisingly capable decoder model released by Alibaba in 2024. It is about four times larger than the GPT-2 model from notebook 1, but still runs on a laptop or on a free Colab GPU. We use the *base* model, which was only pre-trained with causal language modeling, not the *instruct* version that was trained to follow instructions and chat.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()  # eval mode: switch off dropout

# The Scoring Function

The function below computes the log-probability of a `text`, optionally given a `context` that comes before it. It feeds context and text through the model in a *single* forward pass. For every position, the model outputs a probability distribution over the next token, and we look up the probability of the token that actually comes next. (This is the same idea as the `sequence_logprob()` function in notebook 1.)

You don't need to understand every line. What matters: **`score(text)` returns one number: the log-probability of the text according to the model.**

In [ ]:
@torch.no_grad()
def score(text, context=""):
    """Sum of the log-probabilities of all tokens in `text`, given `context`."""
    # without a context, we start with a special start/end-of-text token
    context_ids = tokenizer(context)["input_ids"] if context else [tokenizer.eos_token_id]
    text_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([context_ids + text_ids], device=device)
    logits = model(ids).logits[0, :-1]                       # the logits at position t predict the token at t+1
    log_probs = F.log_softmax(logits.float(), dim=-1)
    targets = ids[0, 1:]
    token_log_probs = log_probs[torch.arange(len(targets)), targets]
    return token_log_probs[len(context_ids) - 1:].sum().item()  # only sum over the tokens of `text`

# 1. Comparing Sentences

Let's compare pairs of sentences that differ only slightly. Higher scores (closer to 0) mean more plausible.

In [ ]:
pairs = [
    ("The wine has notes of cherry and vanilla.", "The wine has notes of gasoline and rubber."),
    ("The cat sat on the mat.", "The mat sat on the cat."),
    ("She has eaten the cake.", "She have eaten the cake."),
    ("Paris is the capital of France.", "Paris is the capital of Germany."),
]

pd.DataFrame(
    [(a, score(a), b, score(b)) for a, b in pairs],
    columns=["Sentence A", "Score A", "Sentence B", "Score B"],
).round(1)

In every pair, the model prefers the plausible sentence. Notice what kinds of knowledge are hidden in these scores:
- **world knowledge about wine** (cherry vs. gasoline),
- **common sense** (cats sit on mats, not the other way around),
- **grammar** (*has eaten* vs. *have eaten*),
- **facts** (Paris is the capital of France).

Nobody taught the model any of this explicitly. It all emerged from learning to predict the next token on huge amounts of text.

⚠️ Scores are only comparable between texts of similar length, because every additional token adds a (negative) log-probability. Longer texts always get lower scores.

# 2. Zero-shot Classification

In Session 2, we fine-tuned an encoder model to classify the **emotions** of tweets. That required thousands of labeled examples and a training run.

With a language model, we can classify texts **without any training** (*zero-shot*): we append each possible label to the text and let the model score the result. The label with the highest score wins. For example, for the tweet *"i feel so lucky today"*, we compare:

- *i feel so lucky today. This person feels* **joy**
- *i feel so lucky today. This person feels* **sadness**
- *i feel so lucky today. This person feels* **anger**
- ...

In [ ]:
emotions = load_dataset("dair-ai/emotion", split="test").shuffle(seed=42).select(range(200))
labels = emotions.features["label"].names
labels

In [ ]:
def classify(text, labels, prompt):
    context = prompt.format(text=text)
    scores = [score(" " + label, context=context) for label in labels]
    return int(np.argmax(scores))

prompt = "{text}\nThis person feels"
print(prompt.format(text=emotions[0]["text"]), "...")

In [ ]:
predictions = [classify(text, labels, prompt) for text in emotions["text"]]

print(f"Accuracy: {accuracy_score(emotions['label'], predictions):.3f}")
print(f"Macro-F1: {f1_score(emotions['label'], predictions, average='macro'):.3f}")

In [ ]:
print(classification_report(emotions["label"], predictions, target_names=labels, zero_division=0))

How does this compare to Session 2?

| Approach | Training data | Accuracy | Macro-F1 |
|---|---|---|---|
| Zero-shot scoring (this notebook) | none | see above | see above |
| DistilBERT feature extraction + logistic regression (2.02) | 16,000 tweets | 0.63 | 0.49 |
| DistilBERT fine-tuned (2.02) | 16,000 tweets | 0.93 | 0.89 |

Without a single labeled example, the language model gets close to the feature-extraction approach, but it is far from the fine-tuned model. Zero-shot classification is great when you have no labeled data at all; when you do, fine-tuning still wins.

(Note: the numbers from Session 2 were computed on the full test set; here we use a sample of 200 tweets to save time.)

Let's try the same idea for the sentiment of movie reviews (the IMDB dataset from notebook 3.02).

In [ ]:
imdb = load_dataset("stanfordnlp/imdb", split="test").shuffle(seed=42).select(range(200))

prompt_imdb = "Review: {text}\nSentiment:"
sentiments = ["negative", "positive"]  # label 0 = negative, 1 = positive
predictions_imdb = [classify(text[:1500], sentiments, prompt_imdb) for text in imdb["text"]]  # we cut long reviews to save time

print(f"Accuracy: {accuracy_score(imdb['label'], predictions_imdb):.3f}")

Zero-shot classification is very sensitive to the exact wording of the prompt and the label words. Try other prompts and labels in the exercises!

# 3. Perplexity: How Surprised Is the Model?

In notebook 3.02, we used **perplexity** to measure how well a model fits a domain. Perplexity is just a normalized version of our score: we divide the log-probability by the number of tokens and take the exponential:

$$\text{Perplexity} = \exp\left(-\frac{1}{n} \sum_{t=1}^{n} \log P(w_t \mid w_1, \dots, w_{t-1})\right)$$

Intuitively, a perplexity of 30 means that the model is, on average, as uncertain as if it had to choose between 30 equally likely tokens at each step. Lower means less surprised.

Let's compare the perplexity of three kinds of text:
- **human-written** movie reviews,
- **word salad:** the same reviews with the words in random order, and
- **model-generated** text: reviews continued by the model itself.

In [ ]:
def perplexity(text):
    n_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return np.exp(-score(text) / n_tokens)

In [ ]:
reviews = load_dataset("stanfordnlp/imdb", split="test").shuffle(seed=1).select(range(20))["text"]
human = [" ".join(review.split()[:60]) for review in reviews]          # first 60 words of each review
salad = [" ".join(random.sample(h.split(), len(h.split()))) for h in human]  # same words, random order

generated = []
for h in human[:10]:
    inputs = tokenizer(" ".join(h.split()[:8]), return_tensors="pt").to(device)  # first 8 words as prompt
    output = model.generate(**inputs, max_new_tokens=60, do_sample=True, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    generated.append(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
print("HUMAN:    ", human[0], "\n")
print("SALAD:    ", salad[0], "\n")
print("GENERATED:", generated[0])

In [ ]:
pd.DataFrame({
    "Text": ["Human-written", "Word salad", "Model-generated"],
    "Median perplexity": [np.median([perplexity(t) for t in texts]) for texts in [human, salad, generated]],
}).round(1)

The word salad contains exactly the same words as the human texts, but the model is far more surprised by it: perplexity is sensitive to word order and grammar, not just to vocabulary.

Even more interesting: text that the model generated itself has a *lower* perplexity than human text. Language models prefer likely tokens, so their texts are less surprising (to them) than what humans write. This is one of the ideas behind tools that try to detect AI-generated text. (In practice, such detectors are unreliable, e.g., because sampling settings and newer models change the picture.)

# 4. Looking Inside: Attention Weights

So far, we treated the model as a black box that outputs probabilities. Let's look inside and visualize the **attention weights**: for every token, how much does it "look at" each of the other tokens when computing its representation?

We switch to the smaller **GPT-2** model from notebook 1, which has 12 layers with 12 attention heads each. To get the attention weights, we must load the model with the plain ("eager") attention implementation, because the faster default implementations do not return them.

In [ ]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2", attn_implementation="eager").to(device).eval()

In [ ]:
sentence = "The animal did not cross the street because it was too tired"
inputs = gpt2_tokenizer(sentence, return_tensors="pt").to(device)
words = [gpt2_tokenizer.decode(i).strip() for i in inputs["input_ids"][0]]

with torch.no_grad():
    attentions = gpt2(**inputs, output_attentions=True).attentions

print(f"{len(attentions)} layers, each with shape {tuple(attentions[0].shape)} = (batch, heads, query position, key position)")

For every layer, we get a tensor with one attention matrix per head. Row $i$ of a matrix shows how much token $i$ attends to each token $j$; each row sums to 1. Let's plot the average over all heads for a few layers.

In [ ]:
def plot_attention(layer, ax):
    weights = attentions[layer][0].mean(dim=0).cpu().numpy()  # average over heads
    ax.imshow(weights, cmap="Greys", vmin=0, vmax=1)
    ax.set_xticks(range(len(words)), words, rotation=90)
    ax.set_yticks(range(len(words)), words)
    ax.set_title(f"Layer {layer + 1}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, layer in zip(axes, [0, 5, 11]):
    plot_attention(layer, ax)
plt.tight_layout()
plt.show()

Two things stand out:

1. **The causal mask:** everything above the diagonal is exactly zero. In a decoder, a token can only attend to itself and to *previous* tokens, never to future ones. This is what makes left-to-right generation possible (and the big difference to encoder models like BERT, where every token sees the whole sentence).
2. **The first token gets a lot of attention.** In many layers, most tokens put most of their attention on the very first token (*The*), although it is not particularly important for the meaning. This phenomenon is called an **attention sink**: the model has learned to "park" attention there when no other token is relevant.

Let's zoom in on the word *it*. Does it attend to *animal*, the word it refers to?

In [ ]:
it_position = words.index("it")
it_attention = np.stack([a[0, :, it_position, : it_position + 1].mean(dim=0).cpu().numpy() for a in attentions])

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(it_attention, cmap="Greys", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(it_position + 1), words[: it_position + 1], rotation=90)
ax.set_yticks(range(len(attentions)), [f"Layer {l + 1}" for l in range(len(attentions))])
ax.set_title('Attention of "it" (average over heads)')
plt.show()

pd.DataFrame(it_attention, columns=words[: it_position + 1], index=[f"Layer {l + 1}" for l in range(len(attentions))]).round(2)

Averaged over heads, *it* mostly attends to the first token (the attention sink), to *because*, and to itself. *animal* only gets a small share.

But averages can hide what individual heads do. Let's search all 12 x 12 = 144 heads for the one in which *it* attends most strongly to *animal*.

In [ ]:
animal_position = words.index("animal")
it_to_animal = torch.stack(attentions)[:, 0, :, it_position, animal_position].cpu().numpy()  # shape (layers, heads)
layer, head = np.unravel_index(it_to_animal.argmax(), it_to_animal.shape)
print(f'Layer {layer + 1}, head {head + 1}: "it" puts {it_to_animal[layer, head]:.0%} of its attention on "animal"')

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(attentions[layer][0, head].cpu().numpy(), cmap="Greys", vmin=0, vmax=1)
ax.set_xticks(range(len(words)), words, rotation=90)
ax.set_yticks(range(len(words)), words)
ax.set_title(f"Layer {layer + 1}, head {head + 1}")
plt.show()

There is a head that links the pronoun *it* to the noun it refers to. Heads like this one specialize in particular relations, and their signal is diluted when we average over all heads.

This is an important lesson: **attention weights are not a simple explanation of what the model "thinks".** A single head can look meaningful, averages can hide such heads, and the representations are further transformed by the feed-forward layers between the attention layers. Interpreting large language models is an active research field (*mechanistic interpretability*).

If you want to explore individual heads interactively, have a look at the [BertViz](https://github.com/jessevig/bertviz) package (`pip install bertviz`), which works with GPT-2 as well:

```python
from bertviz import head_view
head_view(attentions, words)
```

# Exercises

1. **Probe the model:** Write your own sentence pairs. Can you find pairs where the model prefers the "wrong" sentence? Can you find pairs that reveal social biases (e.g., *"The doctor said that he ..."* vs. *"The doctor said that she ..."*)?
2. **Prompt engineering:** Try other prompts and label words for the zero-shot classification (e.g., `'Tweet: "{text}"\nThe emotion expressed in this tweet is'`, or *great*/*terrible* instead of *positive*/*negative* for IMDB). How much do the results change?
3. **Bigger is better?** Repeat the zero-shot classification with a smaller model (`openai-community/gpt2`) or a larger one (e.g., `Qwen/Qwen2.5-1.5B`, if your hardware allows it). How does model size affect accuracy?
4. **Human or machine?** Compute the perplexity of a paragraph you wrote yourself and of a paragraph generated by a chatbot. Which one is more surprising for the model?
5. **Attention:** Change the sentence to *"The animal did not cross the street because it was too wide"*. Now *it* refers to the street. What does the head from above do now? Can you also find a head that always attends to the *previous* token?

In [ ]:
# YOUR CODE GOES HERE